<div style="display:flex; align-items:center; gap:18px; text-align:left">
  <img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/assets/logo_ust.png" width="100">
  <div>
    <p>Diplomado en Inteligencia Artificial para los Negocios</p>
    <p>Facultad de Ingeniería y Negocios</p>
    <p>Módulo 2: Fundamentos de Machine Learning y herramientas Low Code</p>
    <p>Semanas 02 a 05: Ejercicios de repaso 1 (solución comentada)</p>
  </div>
</div>

# Ejercicios de repaso 1 · Clases 2 a 5 — Solución comentada

Este notebook resuelve los ejercicios de repaso 1 de las clases **2 a 5**. Cada ejercicio trae:

1. **El código de la solución**, con comentarios línea a línea redactados de forma clara y directa.
2. **Por qué se hizo así**: la razón de cada decisión, la lectura esperada del resultado y el enlace directo
   a la diapositiva correspondiente del curso para profundizar.
3. En los ejercicios más complejos, un **prompt para pedirle a una IA que les ayude a corregir su código**.
   Está pensado para que la IA **revise lo que ustedes escribieron** y explique el error, no para que
   escriba todo de nuevo.

Úsenlo **después** de intentar cada ejercicio: comparar su intento con la solución es lo que más enseña.

## Preparación del entorno

Instalamos las librerías (la salida queda oculta) y dejamos la configuración general.

In [1]:
%%capture
!pip install -q pycaret==4.0.0a8 plotly ipywidgets pyod "setuptools<81"

In [2]:
%matplotlib inline
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px

# en Colab los datos se leen del repo público; en local, desde ../datasets/
REPO = 'https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/'
BASE = '../datasets/' if os.path.exists('../datasets') else REPO

## Bloque A · Clase 2: conocer y preparar los datos

**Caso:** casas en venta de la Región Metropolitana (`precios_casas_rm.csv`). Cada fila es una casa;
`precio_uf` es su precio de venta. En este bloque **no entrenamos modelos**: conocemos la tabla, revisamos
su calidad y la dejamos lista.

In [3]:
casas = pd.read_csv(BASE + 'precios_casas_rm.csv')
print('Filas y columnas:', casas.shape)
casas.head()

Filas y columnas: (2377, 15)


,latitud,longitud,area_construida,area_total,dormitorios,banos,estacionamiento,bodega,piscina,quincho,precio_uf,comuna,distancia_a_estacion_cercana,estacion_cercana,metro
0,-33.286041,-70.894675,50.0,120.0,3,1,No,No,No,No,1010.260,LAMPA,21898.81094,No metro,No
1,-33.280238,-70.885977,103.0,243.0,3,3,No,No,No,No,827.240,LAMPA,21316.06967,No metro,No
2,-33.280178,-70.885673,46.0,166.0,3,2,No,No,No,No,1144.000,LAMPA,21290.61472,No metro,No
3,-33.279629,-70.885345,103.0,90.0,5,2,Si,Si,Si,Si,741.235,LAMPA,21282.65947,No metro,No
4,-33.280996,-70.882782,70.0,90.0,3,1,Si,Si,Si,Si,888.615,LAMPA,20992.96901,No metro,No


### Ejercicio A1 · Tipos de variables

Muestren cuántos **valores distintos** tiene cada columna. Con eso y con la vista de la tabla,
clasifiquen cada columna como **numérica** (discreta o continua) o **categórica** (nominal u ordinal), e
identifiquen el **target** y las **features**.

In [4]:
casas.nunique()   # pocos valores distintos dan una buena pista de variables discretas o categorías.

latitud                         1947
longitud                        1945
area_construida                  216
area_total                       415
dormitorios                        5
banos                              5
estacionamiento                    2
bodega                             2
piscina                            2
quincho                            2
precio_uf                       1309
comuna                            33
distancia_a_estacion_cercana    1948
estacion_cercana                 104
metro                              2
dtype: int64

**Por qué se hizo así**

- **`nunique()`** cuenta los valores distintos de cada columna. Es una pista, no la respuesta: una
  columna con 2 valores (Si/No) es categórica; una con más de mil valores es numérica continua.
- Clasificación esperada:

| Columna | Tipo |
|:--|:--|
| `precio_uf` | numérica continua · **target** |
| `area_construida`, `area_total`, `distancia_a_estacion_cercana` | numéricas continuas |
| `latitud`, `longitud` | numéricas continuas (coordenadas: un número, pero su sentido es geográfico) |
| `dormitorios`, `banos` | numéricas discretas (se cuentan: no existe 2,5 dormitorios) |
| `estacionamiento`, `bodega`, `piscina`, `quincho`, `metro` | categóricas nominales binarias (Si/No) |
| `comuna`, `estacion_cercana` | categóricas nominales (sin orden natural) |

- En esta tabla **no hay ordinales**. Un ejemplo de ordinal aparece en el bloque C: `ahorros` va de 0 a 4
  y el orden **sí** significa algo.
- La distinción importa porque el modelo trata distinto cada tipo: las categóricas se **codifican**
  (se traducen a números) y los faltantes se rellenan con reglas distintas, como verán en A2.

Para profundizar en la clasificación y codificación de variables, pueden consultar la diapositiva [El tipo define la codificación y la imputación](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/02-datos-preparacion.html#/el-tipo-define-la-codificación-y-la-imputación) de la Clase 2.

### Ejercicio A2 · Valores faltantes e imputación

Cuenten los **faltantes por columna**. Luego, **en una copia** de la tabla, rellenen los faltantes de
`area_construida` con la **mediana** y comprueben que esa columna quedó sin faltantes. ¿Con qué rellenarían
`estacionamiento`, que es Si/No?

In [5]:
print(casas.isna().sum())                    # cuántos datos faltan en cada columna

copia = casas.copy()                          # trabajamos en una copia para no alterar la tabla original
mediana = copia['area_construida'].median()   # usamos la mediana porque no se deja engañar por casas gigantes
copia['area_construida'] = copia['area_construida'].fillna(mediana)

print('Mediana usada:', mediana, 'm²')
print('Faltantes que quedan en area_construida:', copia['area_construida'].isna().sum())

latitud                           0
longitud                          0
area_construida                 142
area_total                        0
dormitorios                       0
banos                             0
estacionamiento                 118
bodega                            0
piscina                           0
quincho                           0
precio_uf                         0
comuna                            0
distancia_a_estacion_cercana      0
estacion_cercana                  0
metro                             0
dtype: int64
Mediana usada: 95.0 m²
Faltantes que quedan en area_construida: 0


**Por qué se hizo así**

- **`isna().sum()`** cuenta los vacíos de cada columna. Solo `area_construida` y `estacionamiento`
  tienen faltantes, y en ambos casos son pocos respecto de las 2.377 casas: se pueden rellenar.
- **`copy()`:** rellenar sobre una copia permite comparar antes y después sin cambiar `casas`, que usan
  los demás ejercicios.
- **Mediana para la numérica:** la mediana no se deja arrastrar por las casas enormes (los *outliers*),
  mientras que el promedio sí.
- **`estacionamiento` es texto (Si/No):** no tiene promedio ni mediana. Lo razonable es la **moda**, el
  valor más frecuente.
- **Cuidado con la interpretación:** rellenar es una **conjetura educada**, no un dato real.
- En la práctica **no hace falta hacerlo a mano**: PyCaret rellena los faltantes al crear el experimento
  (A4). Este ejercicio muestra **qué** hace por dentro.

Para repasar las reglas de imputación por tipo de dato, revisen la diapositiva [Imputación: mediana en números y moda en grupos](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/02-datos-preparacion.html#/imputación-mediana-en-números-y-moda-en-grupos) de la Clase 2.

### Ejercicio A3 · Outliers con un boxplot

Dibujen un **boxplot** de `precio_uf` separado por `metro` (Si/No). Luego ordenen la tabla por precio
y muestren las **5 casas más caras**. Decidan: ¿son errores de datos o casos reales que el modelo debe
aprender?

In [6]:
# un boxplot por grupo: con y sin metro cerca, mostrando solo los puntos fuera de los bigotes
px.box(casas, x='metro', y='precio_uf', points='outliers',
       title='Precio de venta según cercanía a metro').show()

# miramos las 5 casas más caras para ver si los datos tienen sentido o si hay errores
casas.sort_values('precio_uf', ascending=False).head(5)

,latitud,longitud,area_construida,area_total,dormitorios,banos,estacionamiento,bodega,piscina,quincho,precio_uf,comuna,distancia_a_estacion_cercana,estacion_cercana,metro
2101,-33.440095,-70.601625,150.0,400.0,4,3,NaN,Si,Si,Si,15000.0,PROVIDENCIA,585.781767,Ines de Suarez,Si
784,-33.440445,-70.523827,160.0,249.0,5,4,Si,Si,Si,Si,15000.0,LA REINA,3613.074149,No metro,No
1024,-33.381058,-70.545548,108.0,298.0,2,1,Si,Si,Si,Si,15000.0,VITACURA,2683.604313,No metro,No
2238,-33.422719,-70.589376,150.0,200.0,5,2,Si,Si,Si,Si,14990.0,LAS CONDES,402.197669,Cristobal Colon,Si
1551,-33.394460,-70.551930,167.0,267.0,3,3,Si,Si,Si,Si,14990.0,LAS CONDES,1437.831040,Hernando de Magallanes,No


**Por qué se hizo así**

- **Boxplot por grupo:** la caja es el 50% central de los precios y la línea interior, la mediana. Los
  puntos sueltos están más allá de los bigotes: son **candidatos** a outlier, no outliers confirmados.
- **Separar por `metro`** sirve para ver que "atípico" depende del grupo.
- **`sort_values(..., ascending=False).head(5)`** ordena de mayor a menor y muestra las filas completas
  de los casos extremos, para juzgar si el precio es **plausible**.
- **Qué muestra la tabla:** las casas más caras están en comunas de alto valor (La Reina, Vitacura,
  Providencia, Las Condes), algo plausible. Pero hay dos señales que merecen revisión:
  - Varias casas valen **exactamente 15.000 UF** y ninguna supera ese valor. Un número redondo repetido
    justo en el máximo sugiere un **tope** de la búsqueda con que se recolectaron los avisos: las casas
    más caras del mercado probablemente no están en los datos. Es un límite del dataset, no del modelo.
  - Una casa de Vitacura de 108 m², 2 dormitorios y 1 baño a 15.000 UF es poco coherente con las demás:
    vale la pena **verificar el aviso** antes de decidir.
- **Decisión esperada:** las casas caras con superficie y comuna coherentes se **conservan**. Borrarlas
  mejoraría la métrica promedio, pero el modelo fallaría justo con ese segmento cuando aparezca en la
  realidad. Se corrigen o eliminan solo los **errores de datos** confirmados.

Para ver cómo interpretar los bigotes y la política frente a valores atípicos, revisen la diapositiva [El boxplot aísla valores tras los bigotes](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/02-datos-preparacion.html#/el-boxplot-aísla-valores-tras-los-bigotes) de la Clase 2.

**Si su código no funciona: prompt para que una IA les ayude a corregirlo**

Copien el prompt, reemplacen lo que está entre corchetes y péguenlo en su asistente de IA. Pidan una corrección explicada, no una solución nueva: el objetivo es entender qué falló.

```text
Estoy aprendiendo machine learning en Google Colab con pandas y plotly express.
Tengo un DataFrame llamado `casas` con las columnas `precio_uf` (numérica) y `metro` (texto: "Si" o "No").

Quiero: (1) un boxplot de `precio_uf` separado por `metro`, mostrando solo los puntos atípicos,
y (2) ver las 5 casas más caras ordenando la tabla por `precio_uf`.

Este es mi código:
[pegar aquí su código]

Este es el error o el resultado que obtengo:
[pegar aquí el mensaje de error o describir qué se ve mal]

Revisa mi código y explícame en lenguaje simple qué está mal. Propón la corrección mínima, manteniendo mi
código lo más corto posible, y explícame cómo leer los puntos atípicos del boxplot.
```

### Ejercicio A4 · Dejar los datos listos con PyCaret

Creen un `RegressionExperiment` con target `precio_uf`, semilla `session_id=42` y un
**80% de los datos para entrenar** (`train_size=0.8`). Muestren cuántas filas quedaron en entrenamiento y
en prueba. Respondan: ¿qué hizo PyCaret automáticamente y qué decisiones siguen siendo de ustedes?
¿Podrían agregar una columna `precio_por_m2` como predictora? ¿Por qué?

In [7]:
from pycaret.tasks import RegressionExperiment

# preparamos los datos con PyCaret
exp_prep = RegressionExperiment(
    target='precio_uf',   # lo que queremos predecir
    session_id=42,        # semilla fija para que a todos nos dé el mismo resultado
    train_size=0.8,       # 80% para entrenar y 20% guardado para la prueba final
).fit(casas)              # le pasamos la tabla original: PyCaret limpia y codifica por dentro

print('Entrenamiento:', exp_prep.X_train.shape, '| Prueba:', exp_prep.X_test.shape)

Entrenamiento: (1901, 14) | Prueba: (476, 14)


**Por qué se hizo así**

- **`target`** le dice a PyCaret qué columna es la respuesta; el resto pasan a ser features.
- **`session_id=42`** fija la semilla del reparto al azar: cualquiera que ejecute el notebook obtiene la
  misma partición y los mismos resultados (reproducibilidad).
- **`train_size=0.8`** reserva el 20% como **conjunto de prueba**: casas que el modelo no verá al
  entrenar, para evaluarlo después con casos nuevos.
- **Se pasa `casas`, no la copia de A2:** PyCaret **rellena faltantes, codifica las categóricas y separa
  entrenamiento y prueba** por su cuenta.
- **Siguen siendo decisiones humanas:** el EDA, qué hacer con los outliers (A3), qué columnas tienen
  sentido como features y si los datos representan bien el problema.
- **`precio_por_m2`, no:** se calcula **con el precio**, que es justamente lo que queremos predecir.
  Al momento de tasar una casa nueva ese valor no existe. Usarlo es **fuga de datos** (*data leakage*):
  el modelo parecería excelente en el notebook y fallaría en la práctica.

Para revisar el flujo completo de preparación automatizada, consulten la diapositiva [PyCaret automatiza el pipeline de preparación](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/02-datos-preparacion.html#/pycaret-automatiza-el-pipeline-de-preparación) de la Clase 2.

## Bloque B · Clase 3: regresión

**Caso:** el mismo de casas. Ahora sí entrenamos modelos para **predecir `precio_uf`** y aprendemos a leer
sus métricas en UF. Ejecuten la celda de partida: deja listo el experimento (70% entrenamiento,
validación cruzada de 5 pliegues).

In [8]:
from pycaret.tasks import RegressionExperiment

casas = pd.read_csv(BASE + 'precios_casas_rm.csv')
exp_reg = RegressionExperiment(target='precio_uf', session_id=42, train_size=0.7, fold=5).fit(casas)
print('Entrenamiento:', exp_reg.X_train.shape, '| Prueba:', exp_reg.X_test.shape)

Entrenamiento: (1663, 14) | Prueba: (714, 14)


### Ejercicio B1 · Comparar modelos y leer el leaderboard

Comparen **regresión lineal** (`lr`), **árbol de decisión** (`dt`), ***random forest*** (`rf`) y
***gradient boosting*** (`gbr`), ordenando por **MAE**. Muestren la tabla y guarden el mejor modelo en
`mejor_reg`. Respondan: ¿qué modelo gana?, ¿cuánto se equivoca en promedio, en UF?, ¿cuánto peor es la
regresión lineal?

In [9]:
# comparamos las familias vistas en clase, ordenando por error promedio en UF
res_reg = exp_reg.compare_models(
    include=['lr', 'dt', 'rf', 'gbr'],   # solo las familias de la clase, para no demorar
    sort='MAE',                          # ordenamos por error en UF, que es lo que entiende el negocio
)
mejor_reg = res_reg.best                 # el modelo ganador, listo para usar
res_reg.leaderboard

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,rf,696.9419,1.198602e+06,1091.9837,0.9063,0.2795,0.2229
1,gbr,778.7094,1.235128e+06,1108.6040,0.9032,0.2856,0.2354
2,dt,885.1403,2.400975e+06,1540.9709,0.8121,0.3807,0.2653
3,lr,1545.7116,4.138293e+06,2032.8635,0.6770,0.6127,0.5353


**Por qué se hizo así**

- **`include`** limita la comparación a las familias de la clase: el resultado es más rápido y más fácil
  de leer.
- **`sort='MAE'`:** el MAE está en **UF** y se lee directo como "en promedio nos equivocamos por tantas
  UF". Por defecto PyCaret ordena por R², que es una proporción y cuesta más traducir a una decisión.
- **Validación cruzada:** cada cifra de la tabla es el **promedio de 5 pliegues** del entrenamiento, no de
  un solo examen. Por eso el ranking es más confiable.
- **Lectura esperada:** gana *random forest* (`rf`), un **ensamble de árboles**: combina muchos árboles y
  capta relaciones no lineales. La regresión lineal (`lr`) se equivoca más del doble, pero es la más fácil
  de explicar: la pregunta de negocio es si esa diferencia justifica perder simplicidad. Aquí, con un
  error tan distinto, sí.
- **R²:** proporción de la variación del precio que el modelo explica (1 = perfecto, 0 = no mejora al
  promedio). Sirve como contexto, pero la decisión se toma con el error en UF.

Para profundizar en la interpretación del error absoluto medio en el negocio, consulten la diapositiva [MAE: error promedio en unidades del negocio](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/03-regresion.html#/mae-error-promedio-en-unidades-del-negocio) de la Clase 3.

### Ejercicio B2 · ¿Qué variables pesan?

Grafiquen la **importancia de variables** de `mejor_reg`. Respondan: ¿cuáles son las tres variables que
más pesan?, ¿les parece razonable desde el negocio inmobiliario?, ¿significa que esas variables
**causan** el precio?

In [10]:
exp_reg.plot_model(mejor_reg, plot='feature')   # importancia de cada variable según el modelo ganador

**Por qué se hizo así**

- **`plot='feature'`** muestra cuánto usa el modelo cada variable para predecir. Es una lectura del
  modelo, no del mundo.
- **Lectura esperada:** dominan la **superficie** (`area_construida`, `area_total`) y la **ubicación**
  (`longitud`, `latitud`), algo coherente con cómo se tasa una casa. Llama la atención que `comuna`
  pese poco: las coordenadas ya llevan casi toda esa información, con más detalle.
- **Asociación no es causalidad:** que una variable pese significa que **ayuda a predecir**, no que
  cambiarla cambie el precio. Por ejemplo, `longitud` pesa mucho porque en Santiago las comunas de mayor
  valor están hacia el oriente: la coordenada **resume el barrio**; no es que correr una casa unos metros
  la encarezca por sí solo.

Para repasar cómo los algoritmos de ensamble calculan el peso de cada variable, revisen la diapositiva [La importancia revela qué variables pesan más](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/03-regresion.html#/la-importancia-revela-qué-variables-pesan-más) de la Clase 3.

### Ejercicio B3 · El examen final: el conjunto de prueba

Evalúen `mejor_reg` sobre el **conjunto de prueba** y comparen su MAE con el de la tabla de B1.
¿El modelo **generaliza** o hay señales de **sobreajuste**?

In [11]:
pred_reg = exp_reg.predict_model(mejor_reg)   # evaluamos en el 20% de casas que el modelo nunca vio
pred_reg.metrics                              # mismas métricas que en B1, pero sobre datos nuevos

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,RandomForestRegressor,681.612,1.139397e+06,1067.4252,0.9061,0.3194,0.261


**Por qué se hizo así**

- **`predict_model`** sin datos nuevos predice el conjunto de prueba que PyCaret apartó en la celda de
  partida. Es el examen más honesto: se usa **una sola vez**, después de elegir el modelo.
- **Comparar con B1:** el MAE de prueba (cerca de 680 UF) es muy parecido al de la validación cruzada
  (cerca de 700 UF). El modelo **generaliza**: rinde igual con casas nuevas. Si en la prueba el error
  fuera mucho mayor, habría memorizado el entrenamiento (**sobreajuste**).
- **Lectura de negocio:** el MAE de prueba es la cifra que se comunica. Un error de ese tamaño puede ser
  aceptable para una casa de 10.000 UF y demasiado grande para una de 1.500 UF.

Para contrastar el error de entrenamiento frente al error de prueba, revisen la diapositiva [Sobreajuste vs. generalización](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/03-regresion.html#/sobreajuste-vs.-generalización) de la Clase 3 (disponible a continuación de forma interactiva):

In [12]:
# diapositiva interactiva de la clase
from IPython.display import IFrame
IFrame('https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/03-regresion.html#/sobreajuste-vs.-generalización', width='100%', height=480)

### Ejercicio B4 · ¿Dónde se equivoca más el modelo?

Con las predicciones de prueba, creen una columna con el **error absoluto** de cada casa y calculen el
**error promedio según `metro`** (Si/No), junto al precio promedio de cada grupo. ¿Dónde se equivoca más?
¿Qué le dirían a un usuario del modelo?

In [13]:
p = pred_reg.predictions                                          # casas de prueba con su predicción
p['error_abs'] = (p['precio_uf'] - p['prediction_label']).abs()   # cuánto se equivocó en cada casa (en UF)
p.groupby('metro')[['error_abs', 'precio_uf']].mean().round(0)    # error promedio y precio promedio por grupo

,error_abs,precio_uf
metro,,
No,602.0,4249.0
Si,925.0,5416.0


**Por qué se hizo así**

- **`prediction_label`** es el precio que predijo el modelo. Restarlo del precio real y quitar el signo
  con `.abs()` da cuántas UF se equivocó en cada casa.
- **El promedio de `error_abs` es el MAE.** Calcularlo por grupo con `groupby` muestra si el error global
  esconde diferencias.
- **Se agrega `precio_uf`** para poner el error en contexto: 900 UF de error no pesan lo mismo en una casa
  de 4.000 UF que en una de 10.000 UF.
- **Lectura esperada:** las casas **cercanas a metro** tienen más error, y no solo porque sean más caras:
  también es mayor como proporción de su precio (cerca de 17%, contra 14% en las demás).
- **Advertencia de uso:** "para casas cerca de metro, tomen la predicción como una referencia más gruesa y
  compleméntenla con una tasación experta".

Para conectar la métrica global con el análisis segmentado, consulten la diapositiva [Aplicaciones de negocio: costos y salarios](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/03-regresion.html#/aplicaciones-de-negocio-costos-y-salarios) de la Clase 3.

**Si su código no funciona: prompt para que una IA les ayude a corregirlo**

Copien el prompt, reemplacen lo que está entre corchetes y péguenlo en su asistente de IA. Pidan una corrección explicada, no una solución nueva: el objetivo es entender qué falló.

```text
Estoy en Google Colab con pandas y PyCaret 4.0.0a8 (API nueva `pycaret.tasks`, no la v3).
Tengo `pred_reg = exp_reg.predict_model(mejor_reg)`. El DataFrame `pred_reg.predictions` tiene las columnas
originales (entre ellas `precio_uf` y `metro`) más `prediction_label`, que es el precio predicho.

Quiero crear una columna `error_abs` con el error absoluto de cada casa y luego una tabla con el promedio
de `error_abs` y de `precio_uf` para cada valor de `metro`, usando groupby.

Este es mi código:
[pegar aquí su código]

Este es el error o el resultado que obtengo:
[pegar aquí el mensaje de error o la tabla que sale]

Revisa mi código, explícame en lenguaje simple qué está mal y propón la corrección mínima, manteniendo mi
código lo más corto posible. No vuelvas a entrenar el modelo.
```

## Bloque C · Clase 4: clasificación

**Caso nuevo:** solicitudes de crédito de un banco (`credito_aleman.csv`, datos reales de UCI). Cada fila
es una persona que pidió un crédito; `incumplimiento` vale **1** si no lo pagó y **0** si lo pagó. Las
demás columnas ya vienen como números: las de montos y plazos son continuas, y `cuenta_corriente`,
`ahorros` e `historial_crediticio` son **códigos ordinales**.

**La decisión de negocio:** aprobar o rechazar el crédito.

In [14]:
credito = pd.read_csv(BASE + 'credito_aleman.csv')
print('Filas y columnas:', credito.shape)
credito.head()

Filas y columnas: (1000, 9)


,duracion_meses,monto_credito,edad,carga_cuota_pct,creditos_vigentes,cuenta_corriente,ahorros,historial_crediticio,incumplimiento
0,6.0,1169.0,67.0,4.0,2.0,1.0,0.0,4.0,0
1,48.0,5951.0,22.0,2.0,1.0,2.0,1.0,2.0,1
2,12.0,2096.0,49.0,2.0,1.0,0.0,1.0,4.0,0
3,42.0,7882.0,45.0,2.0,1.0,1.0,1.0,2.0,0
4,24.0,4870.0,53.0,3.0,2.0,1.0,1.0,3.0,1


### Ejercicio C1 · El desbalance y la línea base ingenua

Calculen qué proporción de solicitudes terminó en **incumplimiento**. Si un modelo "ingenuo" aprobara
a **todos** (predijera siempre 0), ¿qué *accuracy* tendría? ¿Sería un buen modelo para el banco?

In [15]:
credito['incumplimiento'].value_counts(normalize=True)   # proporción de cada clase para ver el desbalance

incumplimiento
0    0.7
1    0.3
Name: proportion, dtype: float64

**Por qué se hizo así**

- **`value_counts(normalize=True)`** entrega proporciones en vez de conteos: es la forma rápida de ver
  el **desbalance** de clases. Aquí, 70% pagó y 30% no.
- **La línea base ingenua** acierta en todos los que pagaron, así que su *accuracy* es **70%**.
- **Por qué engaña:** ese modelo tiene un accuracy alto, pero **no detecta ni un solo incumplimiento**
  (su *recall* es 0). Para el banco es inútil: aprobaría a todos los que no pagarán.
- **Conclusión:** cualquier modelo serio debe superar claramente esa cifra, y conviene mirar métricas
  que se fijen en la clase minoritaria (recall, precision, AUC).

Para entender por qué el accuracy resulta engañoso cuando una clase domina el dataset, revisen la diapositiva [Con desbalance, el accuracy es un espejismo](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/04-clasificacion.html#/con-desbalance-el-accuracy-es-un-espejismo) de la Clase 4.

### Ejercicio C2 · Preparar y comparar clasificadores

Creen un `ClassificationExperiment` con target `incumplimiento`, `session_id=42`, 70% de
entrenamiento, 5 pliegues **estratificados** y variables **normalizadas**. Comparen `dummy`, `lr`, `dt`,
`rf` y `gbc` ordenando por **AUC**, y guarden el mejor en `mejor_clf`. ¿Qué modelo gana? ¿Cuánto mejora
al `dummy` en AUC y en recall?

In [16]:
from pycaret.tasks import ClassificationExperiment

exp_clf = ClassificationExperiment(
    target='incumplimiento',
    session_id=42,
    train_size=0.7,
    fold=5,
    normalize=True,                  # estandarizamos variables para que la logística compita en igualdad
    fold_strategy='stratifiedkfold', # cada pliegue mantiene el 30% de morosos del total
).fit(credito)

res_clf = exp_clf.compare_models(
    include=['dummy', 'lr', 'dt', 'rf', 'gbc'],   # agregamos dummy para ver cuánto le ganamos a la regla ingenua
    sort='AUC',                                   # con desbalance, ordenamos por AUC y no por accuracy
)
mejor_clf = res_clf.best
res_clf.leaderboard

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,gbc,0.7500,0.7520,0.4429,0.6137,0.5136,0.3518,0.3606
1,rf,0.7529,0.7514,0.4095,0.6404,0.4971,0.3443,0.3605
2,lr,0.6914,0.7123,0.1952,0.4743,0.2746,0.1194,0.1397
3,dt,0.7057,0.6415,0.4810,0.5098,0.4931,0.2867,0.2879
4,dummy,0.7000,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000


**Por qué se hizo así**

- **`normalize=True`** lleva las variables a una escala comparable (z-score). Importa para la logística,
  que compara magnitudes; a los árboles casi no les afecta.
- **`fold_strategy='stratifiedkfold'`** mantiene en cada pliegue la misma proporción de incumplimiento
  que en el total. Con clases desbalanceadas evita pliegues "fáciles" o "difíciles" por azar.
- **Incluir `dummy`** pone la línea base de C1 **dentro** de la misma tabla: se ve su accuracy de 70% con
  recall 0 y AUC 0,5 (no ordena el riesgo mejor que el azar).
- **`sort='AUC'`:** el AUC mide si el modelo **ordena** bien a las personas por riesgo. Es más informativo
  que el accuracy cuando una clase domina.
- **Lectura esperada:** el ganador supera claramente al `dummy` en AUC y en recall, aunque su accuracy
  sea solo unos puntos mayor. Ese contraste es la lección de C1 en números.
- **Empates prácticos:** los dos primeros del ranking quedan a milésimas de AUC. Una diferencia tan chica
  no es evidencia de que uno sea mejor; para elegir entre ellos pesan más la simplicidad o el recall.

Para repasar las cuatro familias algorítmicas de clasificación y sus supuestos, consulten la diapositiva [Cuatro familias resuelven la clasificación](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/04-clasificacion.html#/cuatro-familias-resuelven-la-clasificación) de la Clase 4.

### Ejercicio C3 · La matriz de confusión en lenguaje de negocio

Grafiquen la **matriz de confusión** de `mejor_clf`. Identifiquen los **falsos negativos** y los
**falsos positivos** y expliquen qué significa cada uno para el banco. ¿Cuál de los dos errores es más
caro?

In [17]:
exp_clf.plot_model(mejor_clf, plot='confusion_matrix')   # matriz de confusión sobre las solicitudes de prueba

**Por qué se hizo así**

- **La matriz** cruza lo real (filas) con lo predicho (columnas). La clase positiva es **1 =
  incumplimiento**. Si el número de alguna celda no se lee bien, pasen el cursor por encima.
- **Falso negativo (real 1, predicho 0):** el modelo dijo "pagará", se aprobó el crédito y la persona no
  pagó. El banco **pierde el monto prestado**.
- **Falso positivo (real 0, predicho 1):** el modelo dijo "no pagará" y se rechazó a alguien que sí habría
  pagado. El banco **pierde la ganancia** de ese crédito (y un cliente).
- **El falso negativo es más caro:** perder el capital cuesta bastante más que dejar de ganar los
  intereses. La documentación original de este dataset propone un costo **5 veces mayor** para ese error.
- **Lectura esperada:** con el umbral por defecto (0,5) el modelo deja pasar 43 de los 90 incumplimientos
  de la prueba: casi la mitad. Eso motiva el C4.

Para repasar la anatomía de cuadrantes y las consecuencias operativas de cada celda, revisen la diapositiva [La matriz de confusión](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/04-clasificacion.html#/la-matriz-de-confusión) de la Clase 4 (disponible a continuación de forma interactiva):

In [18]:
# diapositiva interactiva de la clase
from IPython.display import IFrame
IFrame('https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/04-clasificacion.html#/la-matriz-de-confusión', width='100%', height=480)

### Ejercicio C4 · Mover el umbral según el costo de los errores

Primero miren el gráfico de **umbral** de `mejor_clf`. Luego, con las predicciones de prueba, armen la
tabla de aciertos y errores para el umbral **0,5** y para el umbral **0,3**. Suponiendo que un falso
negativo cuesta **5** y un falso positivo cuesta **1**, calculen el **costo total** de cada umbral (pueden
hacer la cuenta a mano). ¿Cuál recomiendan?

In [19]:
exp_clf.plot_model(mejor_clf, plot='threshold').show()   # curva de precisión vs recall al mover el umbral

p = exp_clf.predict_model(mejor_clf).predictions          # tabla de prueba con la probabilidad estimada

p['rechazar_05'] = (p['prediction_score'] >= 0.5).astype(int)   # decisión con el umbral clásico (0.50)
print("Matriz con umbral 0.50 (habitual):")
display(pd.crosstab(p['incumplimiento'], p['rechazar_05']))

p['rechazar_03'] = (p['prediction_score'] >= 0.3).astype(int)   # decisión con umbral más estricto (0.30)
print("Matriz con umbral 0.30 (más estricto):")
display(pd.crosstab(p['incumplimiento'], p['rechazar_03']))

# simulador interactivo para probar otros cortes y ver el costo económico
import ipywidgets as widgets

def simular_costo_credito(umbral=0.30):
    rechazar = (p['prediction_score'] >= umbral).astype(int)
    tabla = pd.crosstab(p['incumplimiento'], rechazar)
    fn = tabla.loc[1, 0] if (1 in tabla.index and 0 in tabla.columns) else 0
    fp = tabla.loc[0, 1] if (0 in tabla.index and 1 in tabla.columns) else 0
    costo = 5 * fn + 1 * fp
    print(f"Umbral seleccionado: {umbral:.2f} | Falsos Negativos (costo 5x): {fn} | Falsos Positivos (costo 1x): {fp}")
    print(f"Costo económico total: 5 * {fn} + 1 * {fp} = {costo} unidades")
    display(tabla)

widgets.interact(simular_costo_credito, umbral=widgets.FloatSlider(value=0.30, min=0.10, max=0.80, step=0.05, description='Umbral:'))

Matriz con umbral 0.50 (habitual):


rechazar_05,0,1
incumplimiento,,
0,180,30
1,43,47


Matriz con umbral 0.30 (más estricto):


rechazar_03,0,1
incumplimiento,,
0,151,59
1,23,67


interactive(children=(FloatSlider(value=0.3, description='Umbral:', max=0.8, min=0.1, step=0.05), Output()), _…

<function __main__.simular_costo_credito(umbral=0.3)>

**Por qué se hizo así**

- **El gráfico de umbral** muestra el intercambio: al bajar el umbral sube el recall (se detectan más
  incumplimientos) y baja la precision (se rechaza a más buenos clientes).
- **`prediction_score`** es la probabilidad de incumplimiento. El modelo entrega un **puntaje de
  riesgo**; el umbral es una **decisión de negocio** que lo convierte en "aprobar" o "rechazar".
- **`>= 0.5` y `astype(int)`** comparan cada probabilidad con el umbral y dejan 1 (rechazar) o 0 (aprobar).
- **`pd.crosstab`** arma la tabla de aciertos y errores: filas = lo real, columnas = la decisión. El falso
  negativo está en la fila 1, columna 0; el falso positivo, en la fila 0, columna 1.
- **La cuenta, con costo total = 5 × FN + 1 × FP:**

| Umbral | Falsos negativos | Falsos positivos | Costo total |
|:--|:--|:--|:--|
| 0,5 | 43 | 30 | 5 × 43 + 30 = **245** |
| 0,3 | 23 | 59 | 5 × 23 + 59 = **174** |

- **Recomendación:** el umbral **0,3**. Rechaza a más buenos clientes, pero evita 20 créditos impagos, y
  como cada impago cuesta 5 veces más, el costo total baja. La recomendación se defiende con el costo, no
  con el accuracy. Los pesos 5 y 1 son un **supuesto**: en un banco real los pondría el área de riesgo.
- **Matiz:** elegir el umbral mirando el conjunto de prueba lo "gasta". En un proyecto real, el umbral se
  elige con validación y la prueba se reserva para el informe final.

Para revisar la fundamentación analítica del balance entre precisión y exhaustividad, consulten la diapositiva [El costo del error determina el umbral óptimo](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/04-clasificacion.html#/el-costo-del-error-determina-el-umbral-óptimo) de la Clase 4.

**Si su código no funciona: prompt para que una IA les ayude a corregirlo**

Copien el prompt, reemplacen lo que está entre corchetes y péguenlo en su asistente de IA. Pidan una corrección explicada, no una solución nueva: el objetivo es entender qué falló.

```text
Estoy en Google Colab con pandas y PyCaret 4.0.0a8 (API nueva `pycaret.tasks`, no la v3).
Tengo `p = exp_clf.predict_model(mejor_clf).predictions`. En `p`, la columna `incumplimiento` es la clase
real (1 = no pagó, 0 = pagó) y `prediction_score` es la probabilidad de incumplimiento.

Quiero crear una columna `rechazar` que valga 1 si `prediction_score` es mayor o igual a un umbral (0.5 y
luego 0.3) y 0 si no, y ver la tabla de aciertos y errores con `pd.crosstab`. Después quiero calcular a
mano el costo total = 5 * falsos negativos + 1 * falsos positivos.

Este es mi código:
[pegar aquí su código]

Este es el error o el resultado que obtengo:
[pegar aquí el mensaje de error o la tabla que sale]

Revisa mi código y explícame en lenguaje simple qué está mal. Indícame en qué celda de la tabla están los
falsos negativos y en cuál los falsos positivos. Propón la corrección mínima, manteniendo mi código lo más
corto posible. No vuelvas a entrenar el modelo.
```

## Bloque D · Clase 5: no supervisado y segmentación

**Caso:** los clientes de telecom de la Clase 5 (`clientes.csv`), **sin la columna `fuga`**: no hay
respuesta correcta que predecir. Esta vez segmentamos con **cuatro** variables numéricas, incluida la
**edad**, que en clase no se usó. Dejamos `plan` fuera del agrupamiento para usarla después, al perfilar.

In [20]:
clientes = pd.read_csv(BASE + 'clientes.csv').drop(columns='fuga')
variables = ['edad', 'antiguedad_meses', 'cargo_mensual', 'llamadas_soporte']
X = clientes[variables]
print('Clientes x variables:', X.shape)
X.head()

Clientes x variables: (2000, 4)


,edad,antiguedad_meses,cargo_mensual,llamadas_soporte
0,19,36,55,1
1,73,1,34,6
2,42,11,16,3
3,31,35,25,1
4,70,5,33,4


### Ejercicio D1 · Por qué estandarizar

Miren el resumen estadístico de `X` (mínimo, máximo y desviación). ¿Qué variable dominaría el
agrupamiento si **no** estandarizan? Luego creen un `ClusteringExperiment` con `session_id=42` y
`normalize=True`.

In [21]:
display(X.describe().round(1))   # rango y dispersión de cada variable

from pycaret.tasks import ClusteringExperiment

exp_clu = ClusteringExperiment(
    session_id=42,      # semilla para que los centros al azar partan igual
    normalize=True,     # todas las variables a media 0 y varianza 1 para que pesen por igual
).fit(X)

,edad,antiguedad_meses,cargo_mensual,llamadas_soporte
count,2000.0,2000.0,2000.0,2000.0
mean,46.5,36.8,51.4,1.4
std,17.0,21.4,23.7,1.3
min,18.0,1.0,12.0,0.0
25%,31.0,18.0,32.0,0.0
50%,47.0,37.0,48.0,1.0
75%,61.0,56.0,65.0,2.0
max,75.0,72.0,130.0,8.0


**Por qué se hizo así**

- **`describe()`** pone las escalas lado a lado: es la evidencia para decidir si estandarizar.
- **k-means agrupa por distancias.** El cargo mensual va de 12 a 130 y las llamadas a soporte, de 0 a 8.
  Sin estandarizar, una diferencia de 10 en el cargo pesaría como 10 llamadas a soporte: el **cargo
  mensual** dominaría y las llamadas casi no influirían, aunque sean importantes para el negocio.
- **`normalize=True`** pone a todas en la misma escala para que **pesen por igual**. En k-means no es
  opcional.
- **`session_id`:** los centros iniciales se eligen al azar; con semilla fija, los grupos salen iguales en
  cada ejecución.
- Aquí no hay `target`: el experimento solo prepara los datos para **descubrir** grupos.

Para entender por qué el escalamiento es un requisito matemático en métodos basados en distancia, revisen la diapositiva [La estandarización previa es obligatoria](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/05-no-supervisado.html#/la-estandarización-previa-es-obligatoria) de la Clase 5.

### Ejercicio D2 · Elegir el número de grupos (k)

Esta celda **ya viene escrita**: ejecútenla. Calcula, para k = 2 a 8, la **inercia** (método del codo)
y el **silhouette** sobre los datos estandarizados. Lean los dos gráficos y la tabla: ¿qué k proponen?
Recuerden que el criterio final es que los grupos sean **accionables**.

In [22]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Xs = StandardScaler().fit_transform(X)        # estandarizamos igual que en normalize=True
ks = list(range(2, 9))
inercia, silueta = [], []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)   # probamos 10 inicios y nos quedamos con el mejor
    inercia.append(km.inertia_)                       # dispersión interna: siempre baja al agregar grupos
    silueta.append(silhouette_score(Xs, km.labels_))  # qué tan separado queda cada cliente de los otros grupos

px.line(x=ks, y=inercia, markers=True, title='Método del codo (Inercia según k)',
        labels={'x': 'Número de grupos (k)', 'y': 'Inercia'}).show()
px.line(x=ks, y=silueta, markers=True, title='Coeficiente de Silhouette según k',
        labels={'x': 'Número de grupos (k)', 'y': 'Silhouette'}).show()
pd.DataFrame({'k': ks, 'inercia': np.round(inercia), 'silhouette': np.round(silueta, 3)})

,k,inercia,silhouette
0,2,6195.0,0.230
1,3,5059.0,0.221
2,4,4257.0,0.229
3,5,3773.0,0.219
4,6,3362.0,0.221
5,7,3130.0,0.217
6,8,2906.0,0.217


**Por qué se hizo así**

- **Por qué esta celda viene escrita:** repetir k-means para varios k es un ciclo de código que no es
  el foco del módulo. Lo central es **leer** las curvas y decidir.
- **Qué hace:** estandariza las variables (lo mismo que `normalize=True`), entrena k-means para cada k
  de 2 a 8 y guarda dos medidas.
- **La inercia siempre baja** al agregar grupos (con tantos grupos como clientes sería 0). Por eso no se
  busca el mínimo, sino el **codo**: el punto desde el cual agregar grupos ya casi no mejora.
- **El silhouette** mide si cada cliente está más cerca de su grupo que del vecino. Es una segunda señal:
  cuando el codo es difuso, ayuda a desempatar.
- **Lectura esperada:** en estos datos el codo es **suave** y el silhouette es **casi plano** (todos los
  valores entre 0,21 y 0,23). Los datos no imponen un k: no hay grupos nítidamente separados, sino
  clientes que varían de forma gradual. Es lo habitual en segmentación de clientes.
- **Decisión:** cuando las curvas no deciden, decide el negocio: se prueban los k candidatos y se elige
  el que entrega grupos que el área comercial pueda **nombrar y tratar distinto**. En D3 se justifica
  k = 4.

Para profundizar en la búsqueda del punto de inflexión y la lectura de inercia, consulten la diapositiva [El método del codo busca el quiebre de inercia](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/05-no-supervisado.html#/el-método-del-codo-busca-el-quiebre-de-inercia) de la Clase 5 (disponible a continuación de forma interactiva):

In [23]:
# diapositiva interactiva de la clase
from IPython.display import IFrame
IFrame('https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/05-no-supervisado.html#/el-método-del-codo-busca-el-quiebre-de-inercia', width='100%', height=480)

### Ejercicio D3 · Segmentar, perfilar y nombrar

Con el k que eligieron, creen el modelo `kmeans`, asignen el segmento a cada cliente y armen el
**perfil** de cada grupo (promedios). Luego cuenten cuántos clientes de cada **plan** hay en cada
segmento. Pongan un **nombre** de negocio a cada grupo y propongan **una acción** por segmento.

In [24]:
res_clu = exp_clu.create_model('kmeans', n_clusters=4)    # en PyCaret 4 se usa n_clusters
segmentos = exp_clu.assign_model(res_clu.pipeline)       # tabla original con la columna 'Cluster' agregada

display(segmentos.groupby('Cluster')[variables].mean().round(1))   # perfil promedio de cada grupo en la escala real

# cruzamos con el plan para chequear si los grupos tienen sentido comercial
pd.crosstab(segmentos['Cluster'], clientes['plan'])

,edad,antiguedad_meses,cargo_mensual,llamadas_soporte
Cluster,,,,
Cluster 0,45.5,9.6,50.9,3.3
Cluster 1,30.6,41.7,43.0,0.9
Cluster 2,50.2,42.7,85.3,1.3
Cluster 3,62.0,43.1,39.3,0.9


plan,Básico,Estándar,Premium
Cluster,,,
Cluster 0,148,131,66
Cluster 1,316,315,32
Cluster 2,0,90,295
Cluster 3,346,247,14


**Por qué se hizo así**

- **`n_clusters`** es el nombre correcto del parámetro en PyCaret 4. Con `num_clusters` (el de la v3 y
  de muchos blogs) PyCaret lo ignora sin avisar y deja siempre 4 grupos.
- **`assign_model(res_clu.pipeline)`** devuelve la tabla original con una columna `Cluster`. Se pasa el
  `.pipeline` porque `create_model` entrega un objeto resultado, no el modelo.
- **Perfil con promedios en la escala original:** los grupos se forman con datos estandarizados, pero se
  describen en unidades del negocio (años, meses, pesos, llamadas).
- **`crosstab` con `plan`:** como `plan` no se usó para agrupar, ver cómo se reparte en cada segmento es
  una forma de **validar** que los grupos tienen sentido.
- **Por qué k = 4:** su silhouette es prácticamente igual al de k = 2 (D2), y con k = 2 solo se separan
  clientes nuevos de antiguos. Con k = 4, además, aparece un grupo de **alto cargo mensual, casi todo
  Premium**, que el plan confirma sin haber participado en el agrupamiento.
- **Una propuesta de nombres y acciones** (leer el perfil de arriba; no hay una única respuesta correcta):

| Rasgo dominante del perfil | Nombre posible | Acción posible |
|:--|:--|:--|
| poca antigüedad y más llamadas a soporte | Nuevos con fricción | acompañamiento en los primeros meses |
| jóvenes, antiguos, cargo bajo y pocas llamadas | Jóvenes estables | ofertas para subir de plan |
| cargo mensual alto, mayoría Premium | Alto valor | atención preferente y retención |
| mayores, antiguos, cargo bajo, casi sin Premium | Leales tradicionales | beneficios por permanencia |

- El nombre resume el rasgo dominante y la acción sale de ese rasgo. El árbitro es la **utilidad de
  negocio**, no una métrica.

Para recordar la diferencia entre predecir una etiqueta conocida y descubrir patrones ocultos, vean la diapositiva [Supervisado predice, no supervisado descubre](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/05-no-supervisado.html#/supervisado-predice-no-supervisado-descubre) de la Clase 5.

**Si su código no funciona: prompt para que una IA les ayude a corregirlo**

Copien el prompt, reemplacen lo que está entre corchetes y péguenlo en su asistente de IA. Pidan una corrección explicada, no una solución nueva: el objetivo es entender qué falló.

```text
Estoy en Google Colab con pandas y PyCaret 4.0.0a8 (API nueva `pycaret.tasks`, no la v3).
Tengo `exp_clu = ClusteringExperiment(session_id=42, normalize=True).fit(X)`, donde `X` tiene 4 columnas
numéricas de clientes, guardadas en la lista `variables`. También tengo el DataFrame `clientes` con la
columna `plan`.

Quiero crear un modelo k-means con 4 grupos, asignar el grupo a cada cliente, ver el promedio de cada
variable por grupo y cruzar los grupos con `plan` usando `pd.crosstab`.

Este es mi código:
[pegar aquí su código]

Este es el error o el resultado que obtengo:
[pegar aquí el mensaje de error o la tabla que sale]

Revisa mi código y explícame en lenguaje simple qué está mal. Ten en cuenta que en PyCaret 4 el número de
grupos se pasa con `n_clusters` y que a `assign_model` se le entrega `.pipeline`. Propón la corrección
mínima, manteniendo mi código lo más corto posible.
```

### Ejercicio D4 · Ver los segmentos en 2D

Resuman las cuatro variables estandarizadas (`Xs`, de D2) en **dos componentes principales** con PCA
y dibujen un gráfico de dispersión coloreado por segmento. ¿Se ven separados los grupos? ¿Qué cuidado hay
que tener al leer esos ejes?

In [25]:
from sklearn.decomposition import PCA

Z = PCA(n_components=2).fit_transform(Xs)   # resumimos las 4 variables en 2 coordenadas para poder graficar
px.scatter(x=Z[:, 0], y=Z[:, 1], color=segmentos['Cluster'], opacity=0.6,
           title='Segmentos en 2D (PCA)',
           labels={'x': 'Componente 1', 'y': 'Componente 2', 'color': 'Segmento'})

**Por qué se hizo así**

- **PCA sobre `Xs`:** se usan los datos estandarizados de D2 porque, igual que k-means, PCA se deja
  dominar por la variable de números más grandes si no se estandariza.
- **Dos componentes** permiten **dibujar** una tabla de cuatro columnas en un plano. El color viene de D3:
  PCA no conoce los segmentos.
- **Por qué no se usa un gráfico de PyCaret:** en la versión del curso, `plot='cluster'` dibuja el
  tamaño de cada grupo (no un plano), y `plot='embedding'` vuelve a agrupar una muestra sin estandarizar,
  así que sus colores no coinciden con los segmentos de D3.
- **Cada eje es una mezcla** de las variables originales, no una variable real del negocio: no se puede
  leer como "edad" o "cargo".
- **Dos ejes no muestran todo:** si dos colores se superponen en el plano, puede ser porque se separan en
  una dirección que el gráfico no muestra. Por eso este gráfico **comunica** la segmentación, pero no
  reemplaza los perfiles de D3.

Para revisar la matemática intuitiva detrás de la proyección ortogonal, consulten la diapositiva [PCA reduce dimensiones preservando la varianza](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/05-no-supervisado.html#/pca-reduce-dimensiones-preservando-la-varianza) de la Clase 5.

### Ejercicio D5 · Clientes fuera del patrón

Creen un `AnomalyExperiment` con `session_id=42` sobre `X` y un modelo `iforest` que marque al **3%**
de los clientes como anómalos. Muestren los **5 más raros** y el **% de anómalos por plan**. ¿Qué harían
con esa lista? ¿Significa que son fraudes?

In [26]:
from pycaret.tasks import AnomalyExperiment

ax = AnomalyExperiment(session_id=42).fit(X)
iso = ax.create_model('iforest', fraction=0.03)   # le pedimos marcar el 3% más raro de la cartera
raros = ax.assign_model(iso.pipeline)             # agrega 'Anomaly' (1 = raro) y 'Anomaly_Score'
raros['plan'] = clientes['plan']                  # sumamos el plan para ver en cuál se concentran

display(raros.sort_values('Anomaly_Score', ascending=False).head(5))   # los 5 clientes más atípicos
(raros.groupby('plan')['Anomaly'].mean() * 100).round(1)               # porcentaje de anómalos en cada plan

,edad,antiguedad_meses,cargo_mensual,llamadas_soporte,Anomaly,Anomaly_Score,plan
353,23,11,100,8,1,0.085045,Premium
1129,25,5,111,6,1,0.083130,Premium
795,72,32,91,8,1,0.076331,Premium
45,70,71,92,5,1,0.073401,Premium
1355,32,1,124,5,1,0.065339,Premium


plan
Básico       1.2
Estándar     1.0
Premium     10.3
Name: Anomaly, dtype: float64

**Por qué se hizo así**

- **`fraction=0.03`** no la descubre el modelo: es un **supuesto** del negocio sobre cuántos casos
  conviene revisar (aquí, 60 de 2.000 clientes).
- **Isolation Forest** aísla cada cliente con preguntas al azar: los raros quedan solos con pocas
  preguntas y reciben un `Anomaly_Score` mayor.
- **Ordenar por `Anomaly_Score`** prioriza la revisión: se parte por los más raros.
- **El promedio de `Anomaly` (1/0) por plan es la proporción de anómalos**; por 100, el porcentaje.
- **Lectura esperada:** **Premium concentra los anómalos**, muy por encima de los otros planes. No es que
  los clientes Premium sean sospechosos: los más raros combinan un **cargo mensual alto** (poco común en
  la base) con **muchas llamadas a soporte**, y ese cargo alto es propio de Premium, un plan minoritario.
  Lo minoritario no es anómalo por sí mismo: actuar sobre la lista sin esta lectura castigaría a un grupo
  entero (equidad). Leída bien, la lista sugiere otra cosa: **clientes que pagan mucho y llaman mucho a
  soporte**, un buen candidato para revisar su experiencia.
- **Anomalía no es fraude:** "raro" puede ser un error de datos, un fraude o el cliente más valioso. La
  acción prudente es una **revisión humana** de la lista, nunca una sanción automática.

Para repasar el principio de aislamiento en particiones aleatorias y las consideraciones éticas sobre falsos sospechosos, revisen las diapositivas [La intuición: lo raro se aísla rápido](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/05-no-supervisado.html#/la-intuición-lo-raro-se-aísla-rápido) y [Usar con cuidado: rareza no es culpa](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/05-no-supervisado.html#/usar-con-cuidado-rareza-no-es-culpa) de la Clase 5.

---

**Síntesis.** En el bloque A conocieron y prepararon una tabla (tipos, faltantes, outliers y fuga de
datos). En B compararon modelos de regresión y leyeron el error en UF, también por segmento. En C vieron
que con desbalance el *accuracy* engaña y que el umbral es una decisión de costos. En D descubrieron
segmentos sin etiqueta, los visualizaron y detectaron casos raros para revisión humana. En todos los
bloques, el código es el medio: lo central es **interpretar y decidir**.

Repasen con las **guías de estudio** y las **tarjetas** de las clases 2 a 5.

## Atribución de datos

**Casas de la Región Metropolitana** (`precios_casas_rm.csv`)

**Creador:** Sebastián Contreras, extracción y preparación para uso docente.
**Fuente:** avisos inmobiliarios de [Enlace Inmobiliario](https://www.enlaceinmobiliario.cl/) y [GoPlaceIt](https://www.goplaceit.com/), obtenidos mediante *web scraping*.
**Licencia:** No declarada para esta compilación; fuentes originales: [Enlace Inmobiliario](https://www.enlaceinmobiliario.cl/) y [GoPlaceIt](https://www.goplaceit.com/).
**Modificación:** columnas estandarizadas, codificación UTF-8 y faltantes didácticos; no se incluyen identificadores personales.

**Solicitudes de crédito** (`credito_aleman.csv`)

**Creador:** Hans Hofmann (Statlog German Credit Data).
**Fuente:** [UCI Statlog German Credit Data](https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data).
**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
**Modificación:** selección de 8 variables, columnas traducidas al español y variables ordinales codificadas como números.

**Clientes de telecom** (`clientes.csv`)

**Creador:** Equipo docente del módulo.
**Procedencia:** Datos generados con [`generar_clientes.py`](../../../scripts/generar_clientes.py), con semilla fija.
**Nota:** Datos sintéticos; no representan a una organización real ni a personas reales.
**Modificación:** en estos ejercicios se excluye la columna `fuga` para el bloque de segmentación.